# 🌎 Pipeline Mundial 2026 — Predictor de partidos (elnine.com.ar)
Notebook consolidado: descarga de datos, cálculo de stats, calibración con resultados reales de octavos, y predictor final con marcadores + % de confianza.

**Orden de ejecución:** correr las celdas de arriba hacia abajo, una sola vez por sesión. Las celdas de descarga (Fixture, Caché) tardan unos minutos por el rate-limit de cortesía hacia la API.

## 1. Setup

In [2]:
import pandas as pd
import numpy as np
import requests
import time
from scipy.stats import poisson

HEADERS_ELNINE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-ES,es;q=0.9",
    "Origin": "https://elnine.com.ar",
    "Sec-Fetch-Site": "same-site",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Dest": "empty",
}

FECHA_INICIO_TORNEO = "2026-06-11"
FECHA_FIN_DESCARGA   = "2026-07-20"   # ajusta a la fecha de hoy cada vez que corras esto


## 2. Descargar el fixture completo del Mundial

In [3]:
def obtener_partidos_fecha(fecha_str):
    """Trae los partidos del Mundial de una fecha puntual (YYYY-MM-DD)."""
    url = f"https://api.elnine.com.ar/schedule?date={fecha_str}"
    headers = {**HEADERS_ELNINE, "Referer": "https://elnine.com.ar/"}
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"Error en {fecha_str}: {e}")
        return []
    return [
        {
            "id": m["id"], "home": m["homeTeam"]["name"], "away": m["awayTeam"]["name"],
            "homeScore": m.get("homeScore"), "awayScore": m.get("awayScore"),
            "status": m.get("status"), "stage": m.get("stage"), "date": fecha_str,
        }
        for m in data.get("matches", [])
        if m.get("tournamentCalendarSlug") == "fifa-world-cup"
    ]


def construir_fixture_mundial(fecha_inicio=FECHA_INICIO_TORNEO, fecha_fin=FECHA_FIN_DESCARGA, pausa=1.0):
    """Recorre todas las fechas del torneo y arma el fixture completo en un DataFrame."""
    fechas = pd.date_range(fecha_inicio, fecha_fin).strftime("%Y-%m-%d").tolist()
    todos = []
    for f in fechas:
        todos.extend(obtener_partidos_fecha(f))
        time.sleep(pausa)
    df = pd.DataFrame(todos)
    print(f"Fixture completo: {len(df)} partidos ({(df['status']=='finished').sum()} finalizados)")
    return df

fixture_mundial = construir_fixture_mundial()
display(fixture_mundial.tail(10))


Fixture completo: 104 partidos (96 finalizados)


,id,home,away,homeScore,awayScore,status,stage,date
94,4w3OOhu4E5,Suiza,Colombia,0.0,0.0,finished,None,2026-07-07
95,IlK5Yw8KyA,Argentina,Egipto,3.0,2.0,finished,None,2026-07-07
96,FPx1R1SJaG,Francia,Marruecos,NaN,NaN,scheduled,None,2026-07-09
97,2Z8ut5k8--,España,Bélgica,NaN,NaN,scheduled,None,2026-07-10
98,hnZHt2-y19,Argentina,Suiza,NaN,NaN,scheduled,None,2026-07-11
99,z24rgqm62U,Noruega,Inglaterra,NaN,NaN,scheduled,None,2026-07-11
100,c59GzqTvnc,,,NaN,NaN,scheduled,None,2026-07-14
101,DY8yYKb7Hu,,,NaN,NaN,scheduled,None,2026-07-15
102,IOH1zprV8u,,,NaN,NaN,scheduled,None,2026-07-18
103,QqgF5EpM-D,,,NaN,NaN,scheduled,None,2026-07-19


## 3. Caché de detalle de partidos (stats por jugador + posición + eventos) y corners
Un solo request por partido a `/match/{id}` trae casi todo. Corners hay que pedirlos aparte en `/match/{id}/stats`.

In [4]:
def obtener_partido_completo(match_id):
    """Detalle completo: alineaciones, stats por jugador (posición, pérdidas de balón,
    duelos, tackles...), eventos, forma reciente y head-to-head."""
    url = f"https://api.elnine.com.ar/match/{match_id}"
    headers = {**HEADERS_ELNINE, "Referer": f"https://elnine.com.ar/partido/{match_id}"}
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()["matchDetail"]
    except Exception as e:
        print(f"Error match {match_id}: {e}")
        return None


def obtener_corners(match_id):
    """Único dato que falta en /match/{id}: corners, hay que pedirlo del endpoint /stats."""
    url = f"https://api.elnine.com.ar/match/{match_id}/stats"
    headers = {**HEADERS_ELNINE, "Referer": f"https://elnine.com.ar/partido/{match_id}"}
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        for item in r.json()["stats"]:
            if item["label"] == "Saques de esquina":
                return item["home"], item["away"]
    except Exception as e:
        print(f"Error corners {match_id}: {e}")
    return 0, 0


def construir_caches(fixture_df, pausa=1.1):
    """Descarga UNA vez el detalle + corners de todos los partidos finalizados."""
    finalizados = fixture_df[fixture_df["status"] == "finished"]
    cache_partidos, cache_corners = {}, {}
    total = len(finalizados)
    for i, (_, row) in enumerate(finalizados.iterrows(), 1):
        mid = row["id"]
        print(f"[{i}/{total}] {row['home']} vs {row['away']}", end="\r")
        cache_partidos[mid] = obtener_partido_completo(mid)
        time.sleep(pausa)
        cache_corners[mid] = dict(zip(["home", "away"], obtener_corners(mid)))
        time.sleep(pausa)
    print(f"\nListo: {sum(v is not None for v in cache_partidos.values())}/{total} partidos cacheados")
    return cache_partidos, cache_corners

cache_partidos, cache_corners = construir_caches(fixture_mundial)


[96/96] Argentina vs Egiptoélgicaerdeegovina
Listo: 96/96 partidos cacheados


## 4. Stats agregadas por equipo
`excluir_match_id` permite dejar un partido fuera de la acumulación — imprescindible para el backtest de la sección 6 (si no lo excluimos, el modelo "vería" el resultado que se supone está prediciendo).

In [5]:
def stats_lineup(lineup):
    """Suma stats de todos los jugadores (titulares + suplentes) de un equipo en un partido."""
    acc = {k: 0 for k in ["fouls_committed", "possession_lost", "recoveries",
                           "tackles_total", "aerial_duels_ok", "shots", "shots_on_target",
                           "interceptions", "offsides", "dispossessed"]}
    for j in lineup.get("players", []) + lineup.get("subs", []):
        s = j.get("stats", {})
        if not s:
            continue
        acc["fouls_committed"] += s.get("foulsCommitted", 0)
        acc["possession_lost"] += s.get("possessionLost", 0)
        acc["recoveries"]      += s.get("recoveries", 0)
        acc["shots"]           += s.get("shots", 0)
        acc["shots_on_target"] += s.get("shotsOnTarget", 0)
        acc["interceptions"]   += s.get("interceptions", 0)
        acc["offsides"]        += s.get("offsides", 0)
        acc["dispossessed"]    += s.get("dispossessed", 0)
        acc["aerial_duels_ok"] += s.get("aerialDuels", {}).get("ok", 0)
        acc["tackles_total"]   += s.get("tackles", {}).get("total", 0)
    return acc


def stats_equipo(equipo, fixture_df, cache_partidos, cache_corners, excluir_match_id=None):
    """Acumula goles, remates, corners, tarjetas, faltas, pérdidas de balón, etc.
    de TODOS los partidos finalizados de un equipo, salvo excluir_match_id."""
    partidos = fixture_df[
        ((fixture_df["home"] == equipo) | (fixture_df["away"] == equipo)) &
        (fixture_df["status"] == "finished") &
        (fixture_df["id"] != excluir_match_id)
    ]

    acc = {k: 0 for k in ["goals_for", "goals_against", "matches_played",
                           "shots_for", "shots_against", "shots_on_for", "shots_on_against",
                           "corners_for", "corners_against", "cards_for", "cards_against",
                           "fouls_for", "fouls_against", "possession_lost_for", "possession_lost_against"]}

    for _, row in partidos.iterrows():
        data = cache_partidos.get(row["id"])
        if data is None:
            continue
        es_local = row["home"] == equipo
        lineup_p = data["homeLineup"] if es_local else data["awayLineup"]
        lineup_r = data["awayLineup"] if es_local else data["homeLineup"]
        sp, sr = stats_lineup(lineup_p), stats_lineup(lineup_r)

        acc["goals_for"]     += data["homeTeam"]["score"] if es_local else data["awayTeam"]["score"]
        acc["goals_against"] += data["awayTeam"]["score"] if es_local else data["homeTeam"]["score"]
        acc["shots_for"]      += sp["shots"];             acc["shots_against"]      += sr["shots"]
        acc["shots_on_for"]   += sp["shots_on_target"];   acc["shots_on_against"]   += sr["shots_on_target"]
        acc["fouls_for"]      += sp["fouls_committed"];   acc["fouls_against"]      += sr["fouls_committed"]
        acc["possession_lost_for"]     += sp["possession_lost"]
        acc["possession_lost_against"] += sr["possession_lost"]

        ck = cache_corners.get(row["id"], {"home": 0, "away": 0})
        acc["corners_for"]     += ck["home"] if es_local else ck["away"]
        acc["corners_against"] += ck["away"] if es_local else ck["home"]

        lado_propio = "home" if es_local else "away"
        lado_rival  = "away" if es_local else "home"
        acc["cards_for"]     += sum(1 for e in data.get("events", []) if e["type"] == "yellow" and e["team"] == lado_propio)
        acc["cards_against"] += sum(1 for e in data.get("events", []) if e["type"] == "yellow" and e["team"] == lado_rival)

        acc["matches_played"] += 1

    return acc


def promedios_torneo(fixture_df, cache_partidos, cache_corners):
    """Promedio por equipo/partido de cada métrica, usando TODOS los equipos del torneo."""
    equipos = pd.concat([fixture_df["home"], fixture_df["away"]]).unique()
    campos = ["shots_for", "shots_on_for", "corners_for", "cards_for", "fouls_for", "goals_for"]
    totales = {c: 0 for c in campos}
    n = 0
    for eq in equipos:
        d = stats_equipo(eq, fixture_df, cache_partidos, cache_corners)
        if d["matches_played"] == 0:
            continue
        for c in campos:
            totales[c] += d[c]
        n += d["matches_played"]
    return {c: v / n for c, v in totales.items()}, n

promedios, n_part = promedios_torneo(fixture_mundial, cache_partidos, cache_corners)
print(f"Promedios por equipo/partido, base {n_part} participaciones:")
for k, v in promedios.items():
    print(f"  {k}: {v:.2f}")


Promedios por equipo/partido, base 192 participaciones:
  shots_for: 12.33
  shots_on_for: 4.16
  corners_for: 4.58
  cards_for: 1.31
  fouls_for: 11.41
  goals_for: 1.46


## 5. Motor de predicción (goles con shrinkage + otros mercados)

In [6]:
def fuerzas_gol(datos, avg_goals_total, peso_muestra=0.6):
    """Ataque/defensa con shrinkage hacia 1.0 (neutral) — evita que un equipo con
    0 goles en contra en pocos partidos quede con 'defensa infinita' en el modelo."""
    ataque_bruto  = (datos["goals_for"]     / datos["matches_played"]) / (avg_goals_total / 2)
    defensa_bruta = (datos["goals_against"] / datos["matches_played"]) / (avg_goals_total / 2)
    ataque  = peso_muestra * ataque_bruto  + (1 - peso_muestra) * 1.0
    defensa = peso_muestra * defensa_bruta + (1 - peso_muestra) * 1.0
    return ataque, defensa


def lambdas_gol(datos_local, datos_visit, avg_goals_total, peso_muestra=0.6):
    at_l, def_l = fuerzas_gol(datos_local, avg_goals_total, peso_muestra)
    at_v, def_v = fuerzas_gol(datos_visit, avg_goals_total, peso_muestra)
    return at_l * def_v * (avg_goals_total / 2), at_v * def_l * (avg_goals_total / 2)


def shrink(valor, promedio, peso=0.4):
    return peso * valor + (1 - peso) * promedio


def matriz_marcadores(lam_local, lam_visit, max_goles=6):
    """Grilla de probabilidad Poisson para cada marcador posible (0-0, 1-0, ... )."""
    filas = []
    for gl in range(max_goles + 1):
        for gv in range(max_goles + 1):
            p = poisson.pmf(gl, lam_local) * poisson.pmf(gv, lam_visit)
            filas.append((gl, gv, p))
    df = pd.DataFrame(filas, columns=["goles_local", "goles_visit", "prob"])
    df["prob"] = df["prob"] / df["prob"].sum()   # normaliza (por truncar en max_goles)
    return df.sort_values("prob", ascending=False).reset_index(drop=True)


## 6. Calibración con resultados reales de octavos (backtest leave-one-out)
Idea: para cada octavo YA jugado, recalculamos la predicción **excluyendo ese partido** de las stats acumuladas (así el modelo no "hace trampa" viendo el resultado que predice), y comparamos la probabilidad que le dio al ganador real. Esto sirve para medir qué tan bien calibrado está el modelo en fase de mata-mata específicamente — que es un contexto distinto a la fase de grupos (más tensión, equipos que se cierran, menos margen de error).

⚠️ Con solo 4 partidos de octavos jugados hasta ahora, esto es una muestra chica — el objetivo es dejar el *framework* listo, no sacar una conclusión definitiva todavía. Cada octavo/cuartos que se juegue, agrégalo a la lista y vuelve a correr esta celda para ir afinando el ajuste con más evidencia.

In [7]:
def buscar_match_id(fixture_df, home, away):
    fila = fixture_df[
        ((fixture_df["home"] == home) & (fixture_df["away"] == away)) |
        ((fixture_df["home"] == away) & (fixture_df["away"] == home))
    ]
    return fila.iloc[0] if len(fila) else None


def backtest_partido(home, away, fixture_df, cache_partidos, cache_corners, avg_goals_total, peso_muestra=0.6):
    """Predice un partido YA jugado excluyéndolo de las stats de ambos equipos, y compara contra el resultado real."""
    fila = buscar_match_id(fixture_df, home, away)
    if fila is None or fila["status"] != "finished":
        print(f"{home} vs {away}: no encontrado o no finalizado")
        return None
    mid = fila["id"]

    datos_l = stats_equipo(fila["home"], fixture_df, cache_partidos, cache_corners, excluir_match_id=mid)
    datos_v = stats_equipo(fila["away"], fixture_df, cache_partidos, cache_corners, excluir_match_id=mid)

    lam_l, lam_v = lambdas_gol(datos_l, datos_v, avg_goals_total, peso_muestra)
    grilla = matriz_marcadores(lam_l, lam_v)
    p_local  = grilla.loc[grilla.goles_local >  grilla.goles_visit, "prob"].sum()
    p_empate = grilla.loc[grilla.goles_local == grilla.goles_visit, "prob"].sum()
    p_visit  = grilla.loc[grilla.goles_local <  grilla.goles_visit, "prob"].sum()

    real_l, real_v = int(fila["homeScore"]), int(fila["awayScore"])
    resultado_real = "local" if real_l > real_v else ("empate" if real_l == real_v else "visit")

    return {
        "partido": f"{fila['home']} {real_l}-{real_v} {fila['away']}",
        "p_local": p_local, "p_empate": p_empate, "p_visit": p_visit,
        "resultado_real": resultado_real,
        "prob_asignada_al_real": {"local": p_local, "empate": p_empate, "visit": p_visit}[resultado_real],
    }


# Octavos ya jugados hasta ahora — agrega más filas a medida que se completen
OCTAVOS_JUGADOS = [
    ("Canadá", "Marruecos"),
    ("Paraguay", "Francia"),
    ("Brasil", "Noruega"),
    ("México", "Inglaterra"),
]

avg_goals_total = promedios["goals_for"] * 2
resultados_backtest = []
for home, away in OCTAVOS_JUGADOS:
    r = backtest_partido(home, away, fixture_mundial, cache_partidos, cache_corners, avg_goals_total)
    if r:
        resultados_backtest.append(r)

df_backtest = pd.DataFrame(resultados_backtest)
display(df_backtest[["partido", "p_local", "p_empate", "p_visit", "resultado_real", "prob_asignada_al_real"]])

brier_model = np.mean([(1 - r["prob_asignada_al_real"]) ** 2 for r in resultados_backtest])
print(f"\nBrier score del modelo puro (más bajo = mejor, 0=perfecto, 0.667=random en 3 salidas): {brier_model:.3f}")


,partido,p_local,p_empate,p_visit,resultado_real,prob_asignada_al_real
0,Canadá 0-3 Marruecos,0.467950,0.250869,0.281181,visit,0.281181
1,Paraguay 0-1 Francia,0.084194,0.163433,0.752372,visit,0.752372
2,Brasil 1-2 Noruega,0.614977,0.191160,0.193864,visit,0.193864
3,México 2-3 Inglaterra,0.495270,0.294540,0.210190,visit,0.210190



Brier score del modelo puro (más bajo = mejor, 0=perfecto, 0.667=random en 3 salidas): 0.463


In [8]:
# --- Buscar el mejor "peso de incertidumbre" para partidos de knockout ---
# Mezclamos la probabilidad del modelo con una distribución más plana (más cerca de 1/3-1/3-1/3),
# para ver si comprimir la confianza del modelo mejora la calibración en mata-mata.

def blend_incertidumbre(p_local, p_empate, p_visit, alpha):
    """alpha=1 -> puro modelo. alpha=0 -> totalmente plano (1/3 cada resultado)."""
    plano = 1/3
    return (alpha*p_local + (1-alpha)*plano,
            alpha*p_empate + (1-alpha)*plano,
            alpha*p_visit + (1-alpha)*plano)

mejores = []
for alpha in np.arange(0.0, 1.01, 0.05):
    briers = []
    for r in resultados_backtest:
        pl, pe, pv = blend_incertidumbre(r["p_local"], r["p_empate"], r["p_visit"], alpha)
        prob_real = {"local": pl, "empate": pe, "visit": pv}[r["resultado_real"]]
        briers.append((1 - prob_real) ** 2)
    mejores.append((alpha, np.mean(briers)))

df_alpha = pd.DataFrame(mejores, columns=["alpha", "brier_score"])
mejor_alpha = df_alpha.loc[df_alpha.brier_score.idxmin(), "alpha"]
print(f"Mejor alpha encontrado con n={len(resultados_backtest)} partidos: {mejor_alpha:.2f}")
print(f"(alpha=1.0 sería 'confiar 100% en el modelo'; valores menores comprimen hacia la incertidumbre)")
display(df_alpha)

# Guardamos el alpha calibrado para usarlo en el predictor final.
# Con tan pocos partidos, un valor por defecto conservador (0.7-0.85) es razonable
# si el grid search da algo demasiado extremo por sobreajuste a 4 datos.
ALPHA_KNOCKOUT = float(np.clip(mejor_alpha, 0.6, 1.0))
print(f"\nALPHA_KNOCKOUT usado en el predictor: {ALPHA_KNOCKOUT}")


Mejor alpha encontrado con n=4 partidos: 0.35
(alpha=1.0 sería 'confiar 100% en el modelo'; valores menores comprimen hacia la incertidumbre)


,alpha,brier_score
0,0.00,0.444444
1,0.05,0.442840
2,0.10,0.441501
3,0.15,0.440428
4,0.20,0.439622
5,0.25,0.439082
6,0.30,0.438808
7,0.35,0.438800
8,0.40,0.439058
9,0.45,0.439583



ALPHA_KNOCKOUT usado en el predictor: 0.6


## 9. Incorporar el ranking FIFA como variable
El ranking FIFA resume años de resultados (no solo 3-4 partidos del torneo), así que es una señal complementaria fuerte — sobre todo para equipos con muestra chica en el Mundial. Lo usamos como un **"tilt"**: inclina los lambdas de gol hacia el equipo mejor rankeado, sin reemplazar el modelo de goles reales que ya teníamos.

Puntos tomados de football-ranking.com (tracker en vivo, ya refleja los resultados del Mundial jugados hasta ahora). Actualiza este diccionario manualmente antes de cada tanda de predicciones si quieres los puntos más frescos — no hay una API estable y gratuita para esto, así que toca mantenerlo a mano.

In [14]:
# Códigos de país FIFA para los equipos del Mundial 2026
FIFA_COUNTRY_CODES = {
    "Argentina": "ARG", "España": "ESP", "Francia": "FRA", "Inglaterra": "ENG",
    "Brasil": "BRA", "Marruecos": "MAR", "Portugal": "POR", "Paises Bajos": "NED",
    "Belgica": "BEL", "Mexico": "MEX", "Colombia": "COL", "Alemania": "GER",
    "Croacia": "CRO", "Italia": "ITA", "Suiza": "SUI", "Estados Unidos": "USA",
    "Japon": "JPN", "Senegal": "SEN", "Uruguay": "URU", "Dinamarca": "DEN",
    "Noruega": "NOR", "Iran": "IRN", "Austria": "AUT", "Egipto": "EGY",
    "Ecuador": "ECU", "Nigeria": "NGA", "Turquia": "TUR", "Australia": "AUS",
    "Argelia": "ALG", "Canada": "CAN", "Costa de Marfil": "CIV", "Corea del Sur": "KOR",
    "Ucrania": "UKR", "Paraguay": "PAR", "Ghana": "GHA", "Jordania": "JOR",
    "Uzbekistan": "UZB", "Catar": "QAT", "Arabia Saudita": "KSA", "Tunez": "TUN",
    "Nueva Zelanda": "NZL", "Haiti": "HAI", "Cabo Verde": "CPV", "Curazao": "CUW",
    "Bosnia y Herzegovina": "BIH", "Sudafrica": "RSA", "RD Congo": "COD",
    "Panama": "PAN",
}

# Alias para que coincidan los nombres tal como los devuelve la API de elnine (con tildes, etc.)
FIFA_ALIASES = {
    "Paises Bajos": "Países Bajos", "Belgica": "Bélgica", "Mexico": "México",
    "Japon": "Japón", "Iran": "IR Iran", "Turquia": "Türkiye",
    "Uzbekistan": "Uzbekistán", "Tunez": "Túnez", "Haiti": "Haití",
    "Sudafrica": "Sudáfrica", "Panama": "Panamá", "Canada": "Canadá",
    "Belgica": "Bélgica",
}


def obtener_puntos_fifa_actuales(country_code):
    """Trae SOLO el registro mas reciente (rankings[0]) de la API oficial de FIFA."""
    url = f"https://inside.fifa.com/api/rankings/by-country?gender=1&countryCode={country_code}&footballType=football&locale=es"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/125.0.0.0 Safari/537.36"}
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        data = r.json()
        mas_reciente = data["rankings"][0]  # el primero es el mas nuevo (ya viene ordenado por PubDate)
        return mas_reciente["DecimalTotalPoints"]
    except Exception as e:
        print(f"Error con {country_code}: {e}")
        return None


def construir_fifa_points_actual(pausa=1.0):
    """Recorre todos los equipos del Mundial y arma el diccionario con puntos FIFA vigentes,
    usando el nombre EXACTO que devuelve el fixture de elnine (con tildes) como llave."""
    puntos = {}
    for nombre_base, code in FIFA_COUNTRY_CODES.items():
        p = obtener_puntos_fifa_actuales(code)
        nombre_final = FIFA_ALIASES.get(nombre_base, nombre_base)
        print(f"{nombre_final} ({code}): {p}")
        if p is not None:
            puntos[nombre_final] = p
        time.sleep(pausa)
    return puntos


# Descarga una sola vez por sesion (tarda ~1 minuto por la pausa entre los ~48 paises)
FIFA_POINTS = construir_fifa_points_actual()
FIFA_POINTS_DEFAULT = 1450.0  # fallback para equipos no listados (selecciones mas debiles del torneo)

def puntos_fifa(equipo):
    return FIFA_POINTS.get(equipo, FIFA_POINTS_DEFAULT)


Argentina (ARG): 1877.27
España (ESP): 1874.71
Francia (FRA): 1870.7
Inglaterra (ENG): 1828.02
Brasil (BRA): 1765.86
Marruecos (MAR): 1755.1
Portugal (POR): 1767.85
Países Bajos (NED): 1753.57
Bélgica (BEL): 1742.24
México (MEX): 1687.48
Colombia (COL): 1698.35
Alemania (GER): 1735.77
Croacia (CRO): 1714.87
Italia (ITA): 1704.73
Suiza (SUI): 1650.06
Estados Unidos (USA): 1671.23
Japón (JPN): 1661.58
Senegal (SEN): 1684.07
Uruguay (URU): 1673.07
Dinamarca (DEN): 1619.47
Noruega (NOR): 1557.44
IR Iran (IRN): 1619.58
Austria (AUT): 1597.4
Egipto (EGY): 1562.37
Ecuador (ECU): 1598.52
Nigeria (NGA): 1585.02
Türkiye (TUR): 1605.73
Australia (AUS): 1579.34
Argelia (ALG): 1571.03
Canadá (CAN): 1559.48
Costa de Marfil (CIV): 1540.87
Corea del Sur (KOR): 1591.63
Ucrania (UKR): 1549.29
Paraguay (PAR): 1505.35
Ghana (GHA): 1346.88
Jordania (JOR): 1387.74
Uzbekistán (UZB): 1458.73
Catar (QAT): 1450.31
Arabia Saudita (KSA): 1423.88
Túnez (TUN): 1476.41
Nueva Zelanda (NZL): 1275.58
Haití (HAI): 1293.

In [15]:
def tilt_fifa(lam_local, lam_visit, home, away, k_fifa=0.5):
    """Inclina los lambdas de gol segun la diferencia de puntos FIFA, preservando
    aproximadamente el total de goles esperado del partido (normaliza por la media geometrica).
    k_fifa=0 -> sin efecto. Valores mas altos = el ranking pesa mas."""
    r_local, r_visit = puntos_fifa(home), puntos_fifa(away)
    factor_local = (r_local / 1650.0) ** k_fifa   # 1650 ~ nivel de referencia "equipo mundialista promedio"
    factor_visit = (r_visit / 1650.0) ** k_fifa
    gm = (factor_local * factor_visit) ** 0.5
    return lam_local * (factor_local / gm), lam_visit * (factor_visit / gm)


### Recalibrar con FIFA incluido (grid search conjunto sobre `alpha` y `k_fifa`)
Extendemos el backtest de la Sección 6 para buscar, a la vez, el mejor peso de incertidumbre knockout (`alpha`) **y** cuánto debería pesar el ranking FIFA (`k_fifa`) — usando los mismos 4 octavos ya jugados como referencia. Mismo aviso de antes: n=4 es chico, esto es un punto de partida que se refina solo con más partidos.

In [16]:
def backtest_partido_fifa(home, away, fixture_df, cache_partidos, cache_corners, avg_goals_total,
                            k_fifa=0.5, peso_muestra=0.6):
    fila = buscar_match_id(fixture_df, home, away)
    if fila is None or fila["status"] != "finished":
        return None
    mid = fila["id"]
    datos_l = stats_equipo(fila["home"], fixture_df, cache_partidos, cache_corners, excluir_match_id=mid)
    datos_v = stats_equipo(fila["away"], fixture_df, cache_partidos, cache_corners, excluir_match_id=mid)

    lam_l, lam_v = lambdas_gol(datos_l, datos_v, avg_goals_total, peso_muestra)
    lam_l, lam_v = tilt_fifa(lam_l, lam_v, fila["home"], fila["away"], k_fifa)

    grilla = matriz_marcadores(lam_l, lam_v)
    p_local  = grilla.loc[grilla.goles_local >  grilla.goles_visit, "prob"].sum()
    p_empate = grilla.loc[grilla.goles_local == grilla.goles_visit, "prob"].sum()
    p_visit  = grilla.loc[grilla.goles_local <  grilla.goles_visit, "prob"].sum()

    real_l, real_v = int(fila["homeScore"]), int(fila["awayScore"])
    resultado_real = "local" if real_l > real_v else ("empate" if real_l == real_v else "visit")
    return {"p_local": p_local, "p_empate": p_empate, "p_visit": p_visit, "resultado_real": resultado_real}


mejores_2d = []
for k_fifa in np.arange(0.0, 1.55, 0.1):
    resultados_k = []
    for home, away in OCTAVOS_JUGADOS:
        r = backtest_partido_fifa(home, away, fixture_mundial, cache_partidos, cache_corners, avg_goals_total, k_fifa=k_fifa)
        if r:
            resultados_k.append(r)
    for alpha in np.arange(0.0, 1.01, 0.1):
        briers = []
        for r in resultados_k:
            pl, pe, pv = blend_incertidumbre(r["p_local"], r["p_empate"], r["p_visit"], alpha)
            prob_real = {"local": pl, "empate": pe, "visit": pv}[r["resultado_real"]]
            briers.append((1 - prob_real) ** 2)
        mejores_2d.append((k_fifa, alpha, np.mean(briers)))

df_grid = pd.DataFrame(mejores_2d, columns=["k_fifa", "alpha", "brier_score"])
mejor_fila = df_grid.loc[df_grid.brier_score.idxmin()]
print(f"Mejor combinacion con n={len(OCTAVOS_JUGADOS)} partidos: k_fifa={mejor_fila.k_fifa:.2f}, alpha={mejor_fila.alpha:.2f}, brier={mejor_fila.brier_score:.3f}")
print(f"(Comparar contra brier sin FIFA: {brier_model:.3f})")

# Igual que antes: con tan pocos partidos, evitamos irnos a extremos por sobreajuste
K_FIFA = float(np.clip(mejor_fila.k_fifa, 0.2, 1.0))
ALPHA_KNOCKOUT = float(np.clip(mejor_fila.alpha, 0.6, 1.0))
print(f"\nUsando en el predictor: K_FIFA={K_FIFA}, ALPHA_KNOCKOUT={ALPHA_KNOCKOUT}")


Mejor combinacion con n=4 partidos: k_fifa=1.50, alpha=0.50, brier=0.427
(Comparar contra brier sin FIFA: 0.463)

Usando en el predictor: K_FIFA=1.0, ALPHA_KNOCKOUT=0.6


## 11. Altitud del estadio y calor (nuevas variables)
Dos ajustes con respaldo real en la literatura de rendimiento deportivo:

- **Altitud**: por encima de ~1500m el rendimiento aerobico baja para equipos no aclimatados (efecto bien documentado, ej. selecciones jugando en Quito o Mexico DF). Aplicamos una penalizacion leve al ataque del equipo NO acostumbrado a la altura.
- **Calor**: partidos al mediodia en sedes calurosas generan mas fatiga y bajan la intensidad, especialmente en el segundo tiempo. Usamos temperatura promedio de julio por ciudad (climatologia, no pronostico exacto — mas honesto que fingir precision que no tenemos para partidos lejanos) combinada con la hora local del partido.

⚠️ Los valores de altitud/temperatura de cada sede son aproximaciones razonables (fuentes publicas), no mediciones oficiales del organizador. Ajustalos si tienes datos mas precisos.

In [24]:
# Estadio -> (altitud_metros, offset_utc_horas, temp_promedio_julio_celsius)
# offset_utc: horas a sumar a UTC para obtener hora local (ej. -6 = UTC menos 6)
ESTADIOS_INFO = {
    "Azteca":            (2240, -6, 24),
    "Akron":             (1566, -6, 28),
    "Guadalajara":       (1566, -6, 28),
    "Monterrey":         (538,  -6, 34),
    "BBVA":              (538,  -6, 34),
    "BC Place":          (0,    -7, 22),
    "Vancouver":         (0,    -7, 22),
    "BMO Field":         (76,   -4, 27),
    "Toronto":           (76,   -4, 27),
    "MetLife":           (3,    -4, 29),
    "New Jersey":        (3,    -4, 29),
    "Lincoln Financial": (12,   -4, 30),
    "Philadelphia":      (12,   -4, 30),
    "Gillette":          (140,  -4, 27),
    "Foxborough":        (140,  -4, 27),
    "Lumen":             (30,   -7, 23),
    "Seattle":           (30,   -7, 23),
    "Levi's":            (30,   -7, 26),
    "Santa Clara":       (30,   -7, 26),
    "SoFi":              (37,   -7, 24),
    "Inglewood":         (37,   -7, 24),
    "Los Angeles":       (37,   -7, 24),
    "Arrowhead":         (317,  -5, 30),
    "Kansas City":       (317,  -5, 30),
    "AT&T":              (180,  -5, 35),
    "Arlington":         (180,  -5, 35),
    "Dallas":            (180,  -5, 35),
    "NRG":               (13,   -5, 34),
    "Houston":           (13,   -5, 34),
    "Hard Rock":         (2,    -4, 32),
    "Miami":             (2,    -4, 32),
    "Mercedes-Benz":     (320,  -4, 32),
    "Atlanta":           (320,  -4, 32),
}

# Equipos del Mundial 2026 con seleccion que juega/entrena habitualmente en altura
# (por eso llegan mas acostumbrados que el resto y no deberian recibir la penalizacion)
EQUIPOS_ACLIMATADOS_ALTURA = {"México", "Ecuador", "Colombia", "Bolivia"}


def info_estadio(stadium_str):
    """Busca coincidencia por substring en el nombre del estadio (formato 'Nombre, Pais')."""
    for clave, info in ESTADIOS_INFO.items():
        if clave.lower() in stadium_str.lower():
            return info
    return (0, -5, 27)  # fallback neutral si no reconoce el estadio


def hora_local_partido(start_time_iso, offset_utc):
    """Convierte el startTime UTC (ej. '2026-07-04T17:00:00Z') a hora local del estadio."""
    from datetime import datetime, timedelta
    dt_utc = datetime.strptime(start_time_iso, "%Y-%m-%dT%H:%M:%SZ")
    dt_local = dt_utc + timedelta(hours=offset_utc)
    return dt_local.hour


In [25]:
def ajustar_altitud_calor(lam_local, lam_visit, home, away, stadium_str, start_time_iso):
    """Ajusta los lambdas de gol por altitud (asimetrico, penaliza al equipo no aclimatado)
    y por calor (simetrico, reduce intensidad de ambos si hace calor y es partido de dia)."""

    altitud, offset_utc, temp_prom = info_estadio(stadium_str)

    # --- Altitud ---
    factor_local, factor_visit = 1.0, 1.0
    if altitud > 1500:
        penalizacion = min(0.15, (altitud - 1500) / 1500 * 0.15)  # hasta -15% en altitud extrema
        if home not in EQUIPOS_ACLIMATADOS_ALTURA:
            factor_local *= (1 - penalizacion)
        if away not in EQUIPOS_ACLIMATADOS_ALTURA:
            factor_visit *= (1 - penalizacion)

    # --- Calor ---
    try:
        hora_local = hora_local_partido(start_time_iso, offset_utc)
    except Exception:
        hora_local = 18  # asume noche si no se puede parsear (sin penalizacion)

    es_horario_caluroso = 11 <= hora_local <= 16
    if es_horario_caluroso and temp_prom >= 30:
        factor_calor = 0.95   # calor fuerte + partido de dia
    elif es_horario_caluroso and temp_prom >= 27:
        factor_calor = 0.975  # calor moderado + partido de dia
    else:
        factor_calor = 1.0

    factor_local *= factor_calor
    factor_visit *= factor_calor

    return lam_local * factor_local, lam_visit * factor_visit, {
        "altitud_m": altitud, "temp_prom_c": temp_prom, "hora_local": hora_local,
        "penalizacion_altura_aplicada": altitud > 1500,
        "penalizacion_calor_aplicada": factor_calor < 1.0,
    }


## 12. Perfil de arbitros (tarjetas y faltas que permiten)
⚠️ Aviso honesto antes de empezar, bro: el arbitro de un partido normalmente se confirma solo **3-5 dias antes**, no con semanas de anticipacion. Esto significa que para predicciones de octavos/cuartos que aun no tienen arbitro asignado, esta variable simplemente no estara disponible todavia — no es que falte scrapear algo, es que el dato no existe aun en ningun lado.

La buena noticia: el campo `referee` ya viene incluido en el `matchDetail` que descargamos en la Seccion 3 — no hace falta pedirle nada nuevo a la API. Armamos el perfil de cada arbitro usando la data que ya esta cacheada, y lo dejamos listo para usar en cuanto se confirme el arbitro de un partido especifico.

In [28]:
def construir_perfil_arbitros(cache_partidos):
    """Agrupa tarjetas y faltas por arbitro, usando SOLO datos ya cacheados (sin requests nuevos)."""
    perfiles = {}

    for match_id, data in cache_partidos.items():
        if data is None:
            continue
        arbitro = data.get("referee")
        if not arbitro:
            continue

        if arbitro not in perfiles:
            perfiles[arbitro] = {"partidos": 0, "tarjetas_totales": 0, "faltas_totales": 0, "rojas_totales": 0}

        tarjetas = sum(1 for e in data.get("events", []) if e["type"] == "yellow")
        rojas = sum(1 for e in data.get("events", []) if e["type"] == "red")

        stats_home = stats_lineup(data["homeLineup"])
        stats_away = stats_lineup(data["awayLineup"])
        faltas = stats_home["fouls_committed"] + stats_away["fouls_committed"]

        perfiles[arbitro]["partidos"] += 1
        perfiles[arbitro]["tarjetas_totales"] += tarjetas
        perfiles[arbitro]["faltas_totales"] += faltas
        perfiles[arbitro]["rojas_totales"] += rojas

    for arbitro, p in perfiles.items():
        p["tarjetas_promedio"] = p["tarjetas_totales"] / p["partidos"]
        p["faltas_promedio"] = p["faltas_totales"] / p["partidos"]
        p["rojas_promedio"] = p["rojas_totales"] / p["partidos"]

    return perfiles


perfil_arbitros = construir_perfil_arbitros(cache_partidos)

# Mostrar ordenado por partidos dirigidos (los que mas partidos tienen dan promedios mas confiables)
df_arbitros = pd.DataFrame(perfil_arbitros).T.sort_values("partidos", ascending=False)
display(df_arbitros)


,partidos,tarjetas_totales,faltas_totales,rojas_totales,tarjetas_promedio,faltas_promedio,rojas_promedio
Adham Makhadmeh,4.0,8.0,70.0,0.0,2.000000,17.500000,0.000000
Wilton Pereira Sampaio,3.0,4.0,74.0,3.0,1.333333,24.666667,1.000000
François Letexier,3.0,13.0,73.0,0.0,4.333333,24.333333,0.000000
Slavko Vinčić,3.0,7.0,71.0,1.0,2.333333,23.666667,0.333333
Yael Falcón Pérez,3.0,4.0,62.0,0.0,1.333333,20.666667,0.000000
Héctor Saíd Martínez Sorto,3.0,8.0,95.0,0.0,2.666667,31.666667,0.000000
Ismail Elfath,3.0,8.0,55.0,1.0,2.666667,18.333333,0.333333
Maurizio Mariani,3.0,9.0,62.0,0.0,3.000000,20.666667,0.000000
Jalal Jayed,3.0,8.0,88.0,0.0,2.666667,29.333333,0.000000
Danny Desmond Makkelie,3.0,9.0,82.0,0.0,3.000000,27.333333,0.000000


### Usar el perfil del arbitro en la prediccion (solo si ya esta confirmado)

In [29]:
def lambda_tarjetas_con_arbitro(datos_local, datos_visit, promedios, arbitro=None, perfil_arbitros=None,
                                  peso_equipo=0.6, peso_arbitro=0.4, factor_ko=1.0):
    """Combina el promedio de tarjetas basado en los EQUIPOS (lo que ya teniamos) con el promedio
    del ARBITRO especifico (si se conoce). Si no hay arbitro confirmado, cae al calculo normal."""

    lam_equipos = (shrink(datos_local["cards_for"] / datos_local["matches_played"], promedios["cards_for"]) +
                   shrink(datos_visit["cards_for"] / datos_visit["matches_played"], promedios["cards_for"]))

    if arbitro and perfil_arbitros and arbitro in perfil_arbitros and perfil_arbitros[arbitro]["partidos"] >= 2:
        lam_arbitro = perfil_arbitros[arbitro]["tarjetas_promedio"]
        lam_final = peso_equipo * lam_equipos + peso_arbitro * lam_arbitro
        fuente = f"blend equipos+arbitro ({arbitro}, {perfil_arbitros[arbitro]['partidos']} partidos dirigidos)"
    else:
        lam_final = lam_equipos
        fuente = "solo equipos (arbitro no confirmado o con muestra insuficiente)"

    return lam_final * factor_ko, fuente


# Ejemplo de uso manual, una vez que el arbitro de un partido se confirme:
# lam_tarjetas, fuente = lambda_tarjetas_con_arbitro(
#     datos_portugal, datos_espana, promedios,
#     arbitro="Michael Oliver", perfil_arbitros=perfil_arbitros, factor_ko=1.20
# )
# print(fuente, "-> lambda:", lam_tarjetas)


## 13. Traer sede, hora y arbitro de partidos AUN NO JUGADOS
`construir_cache_completo` (Seccion 3) solo descarga partidos con `status == "finished"`. Pero `/match/{id}` (sin `/stats`) responde igual para partidos programados que ya tienen sede/arbitro confirmados — como viste en la web para Estados Unidos vs Belgica y Portugal vs Espana. Aqui los traemos sin tocar el resto del pipeline.

In [30]:
def obtener_datos_previos_partido(fixture_df, home, away):
    """Trae sede, hora y arbitro de un partido aunque todavia no se haya jugado
    (usa el mismo endpoint /match/{id}, que responde igual este finalizado o no)."""
    fila = fixture_df[
        ((fixture_df["home"] == home) & (fixture_df["away"] == away)) |
        ((fixture_df["home"] == away) & (fixture_df["away"] == home))
    ]
    if fila.empty:
        print(f"No encontrado en el fixture: {home} vs {away}")
        return None
    match_id = fila.iloc[0]["id"]
    data = obtener_partido_completo(match_id)
    return data


# Si el partido aun no aparecia en tu fixture_mundial (fecha_fin muy corta), extiendelo:
fixture_mundial = construir_fixture_mundial(fecha_inicio="2026-06-11", fecha_fin="2026-07-07")

datos_previos_eeuu_bel = obtener_datos_previos_partido(fixture_mundial, "Estados Unidos", "Bélgica")
datos_previos_por_esp = obtener_datos_previos_partido(fixture_mundial, "Portugal", "España")

for nombre, d in [("Estados Unidos vs Bélgica", datos_previos_eeuu_bel), ("Portugal vs España", datos_previos_por_esp)]:
    if d:
        print(f"{nombre} -> sede: {d.get('stadium')} | arbitro: {d.get('referee')} | hora UTC: {d.get('startTime')}")
    else:
        print(f"{nombre} -> sin datos aun")


Fixture completo: 96 partidos (96 finalizados)
Estados Unidos vs Bélgica -> sede: Seattle Stadium, Estados Unidos | arbitro: Adham Makhadmeh | hora UTC: 2026-07-07T00:00:00Z
Portugal vs España -> sede: Dallas Stadium, Estados Unidos | arbitro: Anthony Taylor | hora UTC: 2026-07-06T19:00:00Z


## 14. Integrar arbitro en TARJETAS y FALTAS (no en remates/corners)
El arbitro decide cuando pitar falta o sacar tarjeta, asi que tiene sentido usarlo ahi. Remates y corners dependen 100% de como juegan los equipos — el arbitro no mete goles ni saca corners, asi que esas dos metricas se quedan calculadas solo con datos de equipo, sin tocar.

`predecir_partido_v3` calculaba tarjetas/faltas ignorando el arbitro (aunque ya lo teniamos armado en la Seccion 12) — `predecir_partido_v4` corrige eso.

In [34]:
def lambda_faltas_con_arbitro(datos_local, datos_visit, promedios, arbitro=None, perfil_arbitros=None,
                                peso_equipo=0.6, peso_arbitro=0.4, factor_ko=1.0):
    """Igual logica que lambda_tarjetas_con_arbitro, pero para faltas."""
    lam_equipos = (shrink(datos_local["fouls_for"] / datos_local["matches_played"], promedios["fouls_for"]) +
                   shrink(datos_visit["fouls_for"] / datos_visit["matches_played"], promedios["fouls_for"]))

    if arbitro and perfil_arbitros and arbitro in perfil_arbitros and perfil_arbitros[arbitro]["partidos"] >= 2:
        lam_arbitro = perfil_arbitros[arbitro]["faltas_promedio"]
        lam_final = peso_equipo * lam_equipos + peso_arbitro * lam_arbitro
        fuente = f"blend equipos+arbitro ({arbitro}, {perfil_arbitros[arbitro]['partidos']} partidos)"
    else:
        lam_final = lam_equipos
        fuente = "solo equipos (arbitro no confirmado o muestra insuficiente)"

    return lam_final * factor_ko, fuente


In [ ]:
def predecir_partido_v4(home, away, fixture_df, cache_partidos, cache_corners, promedios,
                          stadium_str=None, start_time_iso=None, arbitro=None, perfil_arbitros=None,
                          es_knockout=True, peso_muestra_goles=0.6, alpha_incertidumbre=None,
                          k_fifa=0.0, top_n_marcadores=5):
    """Igual que predecir_partido_v3, pero tarjetas y faltas usan el perfil del arbitro
    (si esta confirmado) en vez de solo el promedio de los equipos."""

    datos_l = stats_equipo(home, fixture_df, cache_partidos, cache_corners)
    datos_v = stats_equipo(away, fixture_df, cache_partidos, cache_corners)
    if datos_l["matches_played"] == 0 or datos_v["matches_played"] == 0:
        print(f"Sin datos suficientes para {home} o {away}")
        return None

    avg_goals_total = promedios["goals_for"] * 2
    lam_l, lam_v = lambdas_gol(datos_l, datos_v, avg_goals_total, peso_muestra_goles)
    lam_l, lam_v = tilt_fifa(lam_l, lam_v, home, away, k_fifa)

    info_condiciones = None
    if stadium_str and start_time_iso:
        lam_l, lam_v, info_condiciones = ajustar_altitud_calor(lam_l, lam_v, home, away, stadium_str, start_time_iso)

    grilla = matriz_marcadores(lam_l, lam_v)
    p_local  = grilla.loc[grilla.goles_local >  grilla.goles_visit, "prob"].sum()
    p_empate = grilla.loc[grilla.goles_local == grilla.goles_visit, "prob"].sum()
    p_visit  = grilla.loc[grilla.goles_local <  grilla.goles_visit, "prob"].sum()

    if es_knockout and alpha_incertidumbre is not None:
        p_local, p_empate, p_visit = blend_incertidumbre(p_local, p_empate, p_visit, alpha_incertidumbre)

    over25 = grilla.loc[grilla.goles_local + grilla.goles_visit > 2.5, "prob"].sum()
    btts   = grilla.loc[(grilla.goles_local > 0) & (grilla.goles_visit > 0), "prob"].sum()

    f_ko_cards = 1.20 if es_knockout else 1.0
    f_ko_fouls = 1.15 if es_knockout else 1.0

    # Remates y corners: SOLO equipos (el arbitro no influye aca)
    def lam_mercado(campo, prom_key):
        return (shrink(datos_l[campo] / datos_l["matches_played"], promedios[prom_key]) +
                shrink(datos_v[campo] / datos_v["matches_played"], promedios[prom_key]))

    lam_shots    = lam_mercado("shots_for",    "shots_for")
    lam_shots_on = lam_mercado("shots_on_for", "shots_on_for")
    lam_corners  = lam_mercado("corners_for",  "corners_for")

    # Tarjetas y faltas: equipos + arbitro (si esta confirmado)
    lam_cards, fuente_cards = lambda_tarjetas_con_arbitro(
        datos_l, datos_v, promedios, arbitro=arbitro, perfil_arbitros=perfil_arbitros, factor_ko=f_ko_cards
    )
    lam_fouls, fuente_fouls = lambda_faltas_con_arbitro(
        datos_l, datos_v, promedios, arbitro=arbitro, perfil_arbitros=perfil_arbitros, factor_ko=f_ko_fouls
    )

    print(f"\n{'='*58}\n  {home} (FIFA {puntos_fifa(home):.0f}) vs {away} (FIFA {puntos_fifa(away):.0f})\n{'='*58}")
    if info_condiciones:
        print(f"Sede: {stadium_str} | Altitud: {info_condiciones['altitud_m']}m | "
              f"Temp. prom: {info_condiciones['temp_prom_c']}°C | Hora local: {info_condiciones['hora_local']}:00")
    if arbitro:
        print(f"Arbitro: {arbitro}")
        print(f"  Tarjetas -> {fuente_cards}")
        print(f"  Faltas   -> {fuente_fouls}")

    print(f"\n{home} gana: {p_local:.1%}  |  Empate: {p_empate:.1%}  |  {away} gana: {p_visit:.1%}")
    print(f"Over 2.5: {over25:.1%}  |  BTTS: {btts:.1%}\n")

    print(f"Top {top_n_marcadores} marcadores mas probables:")
    top = grilla.head(top_n_marcadores)
    for _, row in top.iterrows():
        print(f"  {home} {int(row.goles_local)} - {int(row.goles_visit)} {away}   ->  {row.prob:.1%}")

    print(f"\nOtros mercados:")
    for nombre, lam in [("Remates totales", lam_shots), ("Remates al arco", lam_shots_on),
                         ("Corners", lam_corners), ("Tarjetas amarillas", lam_cards), ("Faltas", lam_fouls)]:
        linea = round(lam) - 0.5
        p_over = 1 - poisson.cdf(np.floor(linea), lam)
        print(f"  {nombre:<20} (lambda={lam:.2f}) -> linea {linea}: Over {p_over:.1%} / Under {1-p_over:.1%}")

    return {
        "p_local": p_local, "p_empate": p_empate, "p_visit": p_visit,
        "over25": over25, "btts": btts,
        "lambda_tarjetas": lam_cards, "lambda_faltas": lam_fouls,
        "lambda_remates": lam_shots, "lambda_remates_arco": lam_shots_on,
        "lambda_corners": lam_corners,
        "sede": stadium_str,
        "altitud_m": info_condiciones["altitud_m"] if info_condiciones else None,
        "temp_prom_c": info_condiciones["temp_prom_c"] if info_condiciones else None,
        "hora_local": info_condiciones["hora_local"] if info_condiciones else None,
        "arbitro": arbitro,
        "marcadores_top": [
            {"goles_local": int(row.goles_local), "goles_visit": int(row.goles_visit), "prob": float(row.prob)}
            for _, row in grilla.head(top_n_marcadores).iterrows()
        ],
        "grilla_completa": [
            {"goles_local": int(row.goles_local), "goles_visit": int(row.goles_visit), "prob": float(row.prob)}
            for _, row in grilla.iterrows()
        ],
    }


## 15. Analisis EN VIVO — extrapolar el resto del partido
Idea: combinamos lo que YA paso en el partido (remates, corners, tarjetas, faltas hasta el minuto actual) con lo que esperabamos ANTES del partido (el modelo pre-match), y proyectamos los minutos que faltan. Entre mas avanzado esta el partido, mas peso le damos a lo observado en vivo y menos al promedio pre-match — tiene sentido, porque 45 minutos reales de este partido especifico dicen mas que un promedio historico.

El mismo endpoint `/match/{id}` y `/match/{id}/stats` que ya usamos responden igual estando el partido en vivo — solo que con `period` en `"HT"` o `"2H"` en vez de `"FT"`, y los stats reflejan lo jugado hasta ese momento (`source: "partial"`, como vimos en el primer JSON de Paraguay-Francia).

In [38]:
def obtener_estado_vivo(match_id):
    """Trae marcador actual, minuto y stats parciales de un partido en curso."""
    url_stats = f"https://api.elnine.com.ar/match/{match_id}/stats"
    headers = {**HEADERS_ELNINE, "Referer": f"https://elnine.com.ar/partido/{match_id}"}
    try:
        r = requests.get(url_stats, headers=headers, timeout=10)
        r.raise_for_status()
        data = r.json()
        stats_dict = {item["label"]: {"home": item["home"], "away": item["away"]} for item in data["stats"]}
    except Exception as e:
        print(f"Error stats en vivo {match_id}: {e}")
        return None

    detalle = obtener_partido_completo(match_id)
    if detalle is None:
        return None

    period = detalle.get("period")
    minuto = detalle.get("minute")

    if period == "HT":
        minutos_jugados = 45
    elif minuto:
        minutos_jugados = minuto
    elif period == "FT":
        minutos_jugados = 90
    else:
        minutos_jugados = 1  # evita division por cero si no hay dato

    return {
        "period": period,
        "minutos_jugados": minutos_jugados,
        "score_home": detalle["homeTeam"]["score"],
        "score_away": detalle["awayTeam"]["score"],
        "stats": stats_dict,
    }


In [39]:
def proyectar_partido_vivo(match_id, home, away, lam_local_pre, lam_visit_pre,
                             promedios, n_sim=100_000, peso_maximo_vivo=0.85):
    """Combina lo observado en vivo con la expectativa pre-match, y proyecta el resultado final."""

    estado = obtener_estado_vivo(match_id)
    if estado is None:
        print("No se pudo obtener el estado en vivo del partido.")
        return None

    minutos_jugados = estado["minutos_jugados"]
    minutos_restantes = max(90 - minutos_jugados, 1)
    proporcion_jugada = min(minutos_jugados / 90, 1.0)

    # Peso hacia lo observado en vivo: crece con el tiempo jugado, pero nunca supera peso_maximo_vivo
    # (dejamos un piso de incertidumbre incluso muy avanzado el partido — un solo partido es poca muestra)
    peso_vivo = min(proporcion_jugada, peso_maximo_vivo)
    peso_prematch = 1 - peso_vivo

    print(f"\n{'='*58}")
    print(f"  EN VIVO: {home} {estado['score_home']} - {estado['score_away']} {away}")
    print(f"  Minuto {minutos_jugados} ({estado['period']}) | Peso vivo: {peso_vivo:.0%} | Peso pre-match: {peso_prematch:.0%}")
    print(f"{'='*58}")

    # --- Goles restantes ---
    # Ritmo de gol pre-match, prorrateado a los minutos que faltan
    lam_restante_pre_local = lam_local_pre * (minutos_restantes / 90)
    lam_restante_pre_visit = lam_visit_pre * (minutos_restantes / 90)

    # Ritmo de gol observado en ESTE partido hasta ahora, extrapolado a los minutos que faltan
    goles_hasta_ahora_local = estado["score_home"]
    goles_hasta_ahora_visit = estado["score_away"]
    ritmo_obs_local = goles_hasta_ahora_local / minutos_jugados
    ritmo_obs_visit = goles_hasta_ahora_visit / minutos_jugados
    lam_restante_obs_local = ritmo_obs_local * minutos_restantes
    lam_restante_obs_visit = ritmo_obs_visit * minutos_restantes

    lam_restante_local = peso_prematch * lam_restante_pre_local + peso_vivo * lam_restante_obs_local
    lam_restante_visit = peso_prematch * lam_restante_pre_visit + peso_vivo * lam_restante_obs_visit

    # Monte Carlo SOLO de los goles que faltan, sumados al marcador actual
    goles_restantes_local = np.random.poisson(max(lam_restante_local, 0.01), n_sim)
    goles_restantes_visit = np.random.poisson(max(lam_restante_visit, 0.01), n_sim)

    marcador_final_local = estado["score_home"] + goles_restantes_local
    marcador_final_visit = estado["score_away"] + goles_restantes_visit

    resultados = pd.DataFrame({home: marcador_final_local, away: marcador_final_visit})
    p_local  = (resultados[home] >  resultados[away]).mean()
    p_empate = (resultados[home] == resultados[away]).mean()
    p_visit  = (resultados[home] <  resultados[away]).mean()

    marcador_mas_probable = resultados.value_counts().head(5)

    print(f"\n{home} gana: {p_local:.1%}  |  Empate: {p_empate:.1%}  |  {away} gana: {p_visit:.1%}")
    print(f"\nTop 5 marcadores finales mas probables:")
    for (gl, gv), prob in marcador_mas_probable.items():
        print(f"  {home} {gl} - {gv} {away}   ->  {prob/n_sim:.1%}")

    # --- Otros mercados (remates, corners, tarjetas, faltas): mismo blend ---
    labels_map = {
        "Remates totales": "Total remates", "Remates al arco": "Remates al arco",
        "Corners": "Saques de esquina", "Tarjetas amarillas": "Tarjetas amarillas", "Faltas": "Faltas",
    }
    prom_map = {
        "Remates totales": "shots_for", "Remates al arco": "shots_on_for",
        "Corners": "corners_for", "Tarjetas amarillas": "cards_for", "Faltas": "fouls_for",
    }

    print(f"\nProyeccion de mercados a partido completo:")
    proyecciones = {}
    for nombre, label in labels_map.items():
        actual_home = estado["stats"].get(label, {}).get("home", 0)
        actual_away = estado["stats"].get(label, {}).get("away", 0)
        actual_total = actual_home + actual_away

        prom_key = prom_map[nombre]
        lam_pre_total = promedios[prom_key] * 2  # aproximacion: promedio combinado del torneo

        lam_restante_pre = lam_pre_total * (minutos_restantes / 90)
        ritmo_obs = actual_total / minutos_jugados
        lam_restante_obs = ritmo_obs * minutos_restantes

        lam_restante = peso_prematch * lam_restante_pre + peso_vivo * lam_restante_obs
        proyeccion_final = actual_total + lam_restante
        proyecciones[nombre] = proyeccion_final

        linea = round(proyeccion_final) - 0.5
        p_over = 1 - poisson.cdf(np.floor(linea) - actual_total, max(lam_restante, 0.01)) if linea > actual_total else 1.0
        print(f"  {nombre:<20} actual: {actual_total:<5} proyectado final: {proyeccion_final:.1f}  -> linea {linea}: "
              f"Over {p_over:.1%} / Under {1-p_over:.1%}")

    return {"p_local": p_local, "p_empate": p_empate, "p_visit": p_visit, "proyecciones": proyecciones}


## 16. Proyeccion en vivo DESGLOSADA por equipo (remates al arco/fuera, faltas)
La Seccion 15 proyectaba solo el TOTAL combinado. Aqui separamos por equipo y por tipo de remate (al arco vs fuera), que es mas util para lineas especificas tipo 'remates al arco de Espana' o 'faltas cometidas por Portugal' en vez de solo el agregado del partido.

In [41]:
def proyectar_partido_vivo_desglosado(match_id, home, away, promedios, n_sim=100_000, peso_maximo_vivo=0.85):
    """Igual logica de blend pre-match + observado en vivo, pero separado por equipo
    y por tipo de remate (al arco vs fuera del arco)."""

    estado = obtener_estado_vivo(match_id)
    if estado is None:
        print("No se pudo obtener el estado en vivo del partido.")
        return None

    minutos_jugados = estado["minutos_jugados"]
    minutos_restantes = max(90 - minutos_jugados, 1)
    proporcion_jugada = min(minutos_jugados / 90, 1.0)
    peso_vivo = min(proporcion_jugada, peso_maximo_vivo)
    peso_prematch = 1 - peso_vivo

    print(f"\n{'='*60}")
    print(f"  EN VIVO: {home} {estado['score_home']} - {estado['score_away']} {away}")
    print(f"  Minuto {minutos_jugados} ({estado['period']}) | Peso vivo: {peso_vivo:.0%} | Peso pre-match: {peso_prematch:.0%}")
    print(f"{'='*60}")

    # Stats por equipo, calculados con el mismo criterio que el resto del pipeline
    datos_home = stats_equipo(home, fixture_mundial, cache_partidos, cache_corners)
    datos_away = stats_equipo(away, fixture_mundial, cache_partidos, cache_corners)

    # Cada mercado desglosado: (label de la API, campo pre-match home, campo pre-match away, promedio_key)
    mercados = {
        "Remates al arco": ("Remates al arco", "shots_on_for", "shots_on_for"),
        "Remates fuera":   ("Remates fuera",   None, None),  # no tenemos promedio historico separado para "fuera", se estima como shots_for - shots_on_for
        "Faltas cometidas": ("Faltas", "fouls_for", "fouls_for"),
        "Tarjetas amarillas": ("Tarjetas amarillas", "cards_for", "cards_for"),
        "Saques de esquina": ("Saques de esquina", "corners_for", "corners_for"),
    }

    proyeccion_por_equipo = {}

    for nombre_mercado, (label_api, campo_home, campo_away) in mercados.items():
        actual_home = estado["stats"].get(label_api, {}).get("home", 0)
        actual_away = estado["stats"].get(label_api, {}).get("away", 0)

        # Prom pre-match por equipo (si no hay campo especifico, ej. "remates fuera", se aproxima)
        if campo_home:
            prom_pre_home = shrink(datos_home[campo_home] / datos_home["matches_played"], promedios[campo_home])
            prom_pre_away = shrink(datos_away[campo_away] / datos_away["matches_played"], promedios[campo_away])
        else:
            # Remates fuera = remates totales - remates al arco, aproximado con los promedios ya calculados
            shots_home = shrink(datos_home["shots_for"] / datos_home["matches_played"], promedios["shots_for"])
            shots_on_home = shrink(datos_home["shots_on_for"] / datos_home["matches_played"], promedios["shots_on_for"])
            prom_pre_home = max(shots_home - shots_on_home, 0.5)

            shots_away = shrink(datos_away["shots_for"] / datos_away["matches_played"], promedios["shots_for"])
            shots_on_away = shrink(datos_away["shots_on_for"] / datos_away["matches_played"], promedios["shots_on_for"])
            prom_pre_away = max(shots_away - shots_on_away, 0.5)

        # Blend: ritmo pre-match prorrateado + ritmo observado en ESTE partido, extrapolados a lo que falta
        lam_restante_pre_home = prom_pre_home * (minutos_restantes / 90)
        ritmo_obs_home = actual_home / minutos_jugados
        lam_restante_obs_home = ritmo_obs_home * minutos_restantes
        lam_restante_home = peso_prematch * lam_restante_pre_home + peso_vivo * lam_restante_obs_home
        proyeccion_home = actual_home + lam_restante_home

        lam_restante_pre_away = prom_pre_away * (minutos_restantes / 90)
        ritmo_obs_away = actual_away / minutos_jugados
        lam_restante_obs_away = ritmo_obs_away * minutos_restantes
        lam_restante_away = peso_prematch * lam_restante_pre_away + peso_vivo * lam_restante_obs_away
        proyeccion_away = actual_away + lam_restante_away

        proyeccion_por_equipo[nombre_mercado] = {
            "home_actual": actual_home, "home_proyectado": proyeccion_home, "home_lambda_restante": lam_restante_home,
            "away_actual": actual_away, "away_proyectado": proyeccion_away, "away_lambda_restante": lam_restante_away,
        }

    print(f"\n{'Mercado':<20}{'Equipo':<12}{'Actual':<8}{'Proyectado':<12}{'Linea sugerida'}")
    for nombre_mercado, datos in proyeccion_por_equipo.items():
        for lado, nombre_equipo in [("home", home), ("away", away)]:
            actual = datos[f"{lado}_actual"]
            proyectado = datos[f"{lado}_proyectado"]
            lam_restante = datos[f"{lado}_lambda_restante"]
            linea = round(proyectado) - 0.5

            if linea > actual:
                p_over = 1 - poisson.cdf(np.floor(linea) - actual, max(lam_restante, 0.01))
            else:
                p_over = 1.0  # ya se paso la linea con lo jugado hasta ahora

            print(f"{nombre_mercado:<20}{nombre_equipo:<12}{actual:<8}{proyectado:<12.1f}"
                  f"{linea} -> Over {p_over:.1%} / Under {1-p_over:.1%}")

    return proyeccion_por_equipo


## Recalibración honesta (walk-forward): aprender solo de lo que ya paso, nunca del futuro

En vez de calibrar y validar con los mismos 4-5 partidos de la ronda anterior (que sobreajusta facil con tan poca muestra), esto recorre TODOS los partidos ya jugados en orden cronologico. Para cada partido, calcula las stats de ambos equipos usando SOLO los partidos anteriores a su fecha (nunca ve el resultado de ese partido ni de ninguno futuro), predice, y compara contra lo que realmente paso.

Esto reproduce exactamente la progresion que describiste: fase de grupos predice dieciseisavos, se valida, se suma esa info para predecir octavos, se valida, se suma para predecir cuartos — y dentro de cuartos mismo, en cuanto termine Francia vs Marruecos (hoy), ese resultado ya queda disponible para afinar las predicciones de España-Belgica, Noruega-Inglaterra y Argentina-Suiza (que juegan despues). Todo esto sale solo por comparar fechas, sin tener que programar cada ronda a mano.

Ademas del Brier score, se agrega el **Ranked Probability Score (RPS)** — mejor metrica para resultados con 3 categorias ordenadas (visita < empate < local), porque penaliza menos un error "cercano" (decir empate y gano el local) que uno "lejano" (decir que gana la visita por goleada y gano el local).

In [ ]:
def stats_equipo_hasta_fecha(equipo, fixture_df, cache_partidos, cache_corners, fecha_corte):
    """Igual que stats_equipo, pero SOLO usa partidos con fecha ANTERIOR a fecha_corte.
    Esto es lo que garantiza que la validacion sea honesta: nunca usa informacion
    que no estaria disponible en el momento real de esa prediccion."""
    partidos = fixture_df[
        ((fixture_df["home"] == equipo) | (fixture_df["away"] == equipo)) &
        (fixture_df["status"] == "finished") &
        (fixture_df["date"] < fecha_corte)
    ]

    acc = {k: 0 for k in ["goals_for", "goals_against", "matches_played",
                           "shots_for", "shots_against", "shots_on_for", "shots_on_against",
                           "corners_for", "corners_against", "cards_for", "cards_against",
                           "fouls_for", "fouls_against", "possession_lost_for", "possession_lost_against"]}

    for _, row in partidos.iterrows():
        data = cache_partidos.get(row["id"])
        if data is None:
            continue
        es_local = row["home"] == equipo
        lineup_p = data["homeLineup"] if es_local else data["awayLineup"]
        lineup_r = data["awayLineup"] if es_local else data["homeLineup"]
        sp, sr = stats_lineup(lineup_p), stats_lineup(lineup_r)

        acc["goals_for"]     += data["homeTeam"]["score"] if es_local else data["awayTeam"]["score"]
        acc["goals_against"] += data["awayTeam"]["score"] if es_local else data["homeTeam"]["score"]
        acc["shots_for"]      += sp["shots"];             acc["shots_against"]      += sr["shots"]
        acc["shots_on_for"]   += sp["shots_on_target"];   acc["shots_on_against"]   += sr["shots_on_target"]
        acc["fouls_for"]      += sp["fouls_committed"];   acc["fouls_against"]      += sr["fouls_committed"]
        acc["possession_lost_for"]     += sp["possession_lost"]
        acc["possession_lost_against"] += sr["possession_lost"]

        ck = cache_corners.get(row["id"], {"home": 0, "away": 0})
        acc["corners_for"]     += ck["home"] if es_local else ck["away"]
        acc["corners_against"] += ck["away"] if es_local else ck["home"]

        lado_propio = "home" if es_local else "away"
        lado_rival  = "away" if es_local else "home"
        acc["cards_for"]     += sum(1 for e in data.get("events", []) if e["type"] == "yellow" and e["team"] == lado_propio)
        acc["cards_against"] += sum(1 for e in data.get("events", []) if e["type"] == "yellow" and e["team"] == lado_rival)

        acc["matches_played"] += 1

    return acc


def rps_3_resultados(p_local, p_empate, p_visit, resultado_real):
    """Ranked Probability Score para 3 categorias ORDENADAS: visita(0) < empate(1) < local(2).
    0 = prediccion perfecta. Penaliza menos los errores "cercanos" que los "lejanos"."""
    orden = {"visit": 0, "empate": 1, "local": 2}
    probs = [p_visit, p_empate, p_local]
    outcome = [0, 0, 0]
    outcome[orden[resultado_real]] = 1
    cum_p = cum_o = 0
    rps = 0
    for i in range(3):
        cum_p += probs[i]
        cum_o += outcome[i]
        rps += (cum_p - cum_o) ** 2
    return rps / 2  # normalizado: (numero_categorias - 1) = 2


In [ ]:
def precomputar_walkforward(fixture_df, cache_partidos, cache_corners, promedios, min_partidos_previos=1):
    """Recorre TODOS los partidos finalizados en orden cronologico y precalcula los lambdas
    de gol SIN tilt de FIFA (para poder probar distintos k_fifa/alpha rapido despues, sin
    recalcular las stats de equipo cada vez -- eso es lo lento, esto lo hace una sola vez).
    Salta partidos donde algun equipo tiene menos de min_partidos_previos jugados (muestra muy chica)."""
    finalizados = fixture_df[fixture_df["status"] == "finished"].sort_values("date")
    avg_goals_total = promedios["goals_for"] * 2
    precomputado = []

    for _, row in finalizados.iterrows():
        home, away, fecha = row["home"], row["away"], row["date"]
        datos_l = stats_equipo_hasta_fecha(home, fixture_df, cache_partidos, cache_corners, fecha)
        datos_v = stats_equipo_hasta_fecha(away, fixture_df, cache_partidos, cache_corners, fecha)
        if datos_l["matches_played"] < min_partidos_previos or datos_v["matches_played"] < min_partidos_previos:
            continue

        lam_l_raw, lam_v_raw = lambdas_gol(datos_l, datos_v, avg_goals_total, peso_muestra=0.6)
        real_l, real_v = int(row["homeScore"]), int(row["awayScore"])
        resultado_real = "local" if real_l > real_v else ("empate" if real_l == real_v else "visit")

        precomputado.append({
            "home": home, "away": away, "fecha": fecha, "round": row.get("stage"),
            "lam_l_raw": lam_l_raw, "lam_v_raw": lam_v_raw,
            "real_l": real_l, "real_v": real_v, "resultado_real": resultado_real,
        })

    print(f"Partidos usables para walk-forward: {len(precomputado)} de {len(finalizados)} finalizados "
          f"(el resto se salto por muestra insuficiente de alguno de los 2 equipos)")
    return precomputado


def evaluar_walkforward(precomputado, k_fifa, alpha_incertidumbre):
    """Aplica tilt de FIFA y compresion de incertidumbre a los lambdas ya precalculados,
    y devuelve el detalle partido por partido (no solo el promedio) para poder inspeccionar."""
    filas = []
    for p in precomputado:
        lam_l, lam_v = tilt_fifa(p["lam_l_raw"], p["lam_v_raw"], p["home"], p["away"], k_fifa)
        grilla = matriz_marcadores(lam_l, lam_v)
        p_local  = grilla.loc[grilla.goles_local >  grilla.goles_visit, "prob"].sum()
        p_empate = grilla.loc[grilla.goles_local == grilla.goles_visit, "prob"].sum()
        p_visit  = grilla.loc[grilla.goles_local <  grilla.goles_visit, "prob"].sum()
        if alpha_incertidumbre is not None:
            p_local, p_empate, p_visit = blend_incertidumbre(p_local, p_empate, p_visit, alpha_incertidumbre)

        prob_real = {"local": p_local, "empate": p_empate, "visit": p_visit}[p["resultado_real"]]
        filas.append({
            "partido": f"{p['home']} {p['real_l']}-{p['real_v']} {p['away']}",
            "round": p["round"], "fecha": p["fecha"],
            "p_local": p_local, "p_empate": p_empate, "p_visit": p_visit,
            "resultado_real": p["resultado_real"],
            "brier": (1 - prob_real) ** 2,
            "rps": rps_3_resultados(p_local, p_empate, p_visit, p["resultado_real"]),
        })
    return pd.DataFrame(filas)

precomputado_wf = precomputar_walkforward(fixture_mundial, cache_partidos, cache_corners, promedios)


In [ ]:
# Grid search walk-forward: usa TODOS los partidos disponibles (no solo la ronda anterior),
# minimizando RPS promedio (metrica principal para resultados 3-vias ordenados).
resultados_grid = []
for k_fifa_try in np.arange(0.0, 1.55, 0.1):
    for alpha_try in np.arange(0.3, 1.01, 0.1):
        df_try = evaluar_walkforward(precomputado_wf, k_fifa_try, alpha_try)
        resultados_grid.append({
            "k_fifa": k_fifa_try, "alpha": alpha_try,
            "rps_mean": df_try["rps"].mean(), "brier_mean": df_try["brier"].mean(),
        })

df_grid_wf = pd.DataFrame(resultados_grid)
mejor_wf = df_grid_wf.loc[df_grid_wf.rps_mean.idxmin()]

print(f"Mejor combinacion (walk-forward, n={len(precomputado_wf)} partidos): "
      f"k_fifa={mejor_wf.k_fifa:.2f}, alpha={mejor_wf.alpha:.2f}, "
      f"RPS={mejor_wf.rps_mean:.3f}, Brier={mejor_wf.brier_mean:.3f}")
print(f"(Valores previos: K_FIFA={K_FIFA:.2f}, ALPHA_KNOCKOUT={ALPHA_KNOCKOUT:.2f})")

K_FIFA = float(np.clip(mejor_wf.k_fifa, 0.15, 1.1))
ALPHA_KNOCKOUT = float(np.clip(mejor_wf.alpha, 0.3, 1.0))
print(f"
Actualizado -> K_FIFA={K_FIFA}, ALPHA_KNOCKOUT={ALPHA_KNOCKOUT}")

# Reporte de calidad DESGLOSADO POR RONDA -- esto es clave para detectar si el modelo
# predice bien parejo en todo el torneo, o si se le complica alguna ronda en particular
df_final_wf = evaluar_walkforward(precomputado_wf, K_FIFA, ALPHA_KNOCKOUT)
df_final_wf["round"] = df_final_wf["round"].fillna("Sin clasificar (stage vacio en la API)")
reporte_rondas = df_final_wf.groupby("round").agg(
    n=("brier", "count"), brier_prom=("brier", "mean"), rps_prom=("rps", "mean")
).round(3)
print("
Calidad de prediccion por ronda (mas bajo = mejor):")
display(reporte_rondas)


### Chequeo de sensatez (sanity check)
Antes de confiar en una prediccion, desconfia de numeros extremos en partidos de eliminacion directa entre equipos que ya clasificaron -- ahi ya no deberia haber diferencias de nivel gigantes. Si el modelo te da 85%+ de probabilidad a un equipo en cuartos/semis, revisa primero si hay un dato mal cargado (FIFA points, lambda de gol inflado) antes de asumir que es un favoritismo real tan grande. El siguiente chequeo automatico marca cualquier prediccion de `resultados_ronda` que caiga en ese rango.

In [ ]:
def chequeo_sensatez(resultados_dict, umbral=0.75):
    alertas = []
    for partido, r in resultados_dict.items():
        if r is None:
            continue
        p_max = max(r["p_local"], r["p_empate"], r["p_visit"])
        if p_max >= umbral:
            alertas.append(f"  {partido}: probabilidad maxima {p_max:.1%} -- revisar antes de confiar")
    if alertas:
        print(f"ALERTAS: {len(alertas)} partido(s) con probabilidad sospechosamente alta:")
        for a in alertas:
            print(a)
    else:
        print("Sin alertas: ninguna prediccion supera el umbral de sensatez.")

# Corre esto DESPUES de calcular resultados_ronda (celda de "Predicciones de la siguiente ronda")
# chequeo_sensatez(resultados_ronda)


## Predicciones de la siguiente ronda
⚠️ **EDITA la celda de abajo antes de correr**: cambia `CRUCES_RONDA_ACTUAL` por los cruces reales (ej. las semifinales), y `FECHA_FIN_FIXTURE`/`ROUND_ACTUAL` segun corresponda. Todo lo demas corre igual, sin importar la ronda.

In [ ]:
# EDITA ESTAS 3 LINEAS cada vez que avances de ronda. Verifica el ROUND_ACTUAL correcto
# con: fixture_mundial[fixture_mundial["home"]=="Equipo"][["home","away","round" if "round" in fixture_mundial.columns else "stage"]]
ROUND_ACTUAL = "Quarterfinal"  # confirmado: Francia-Marruecos (9/7), España-Bélgica (10/7), Noruega-Inglaterra y Argentina-Suiza (11/7) son los CUARTOS reales
FECHA_FIN_FIXTURE = "2026-08-15"  # cualquier fecha posterior al fin real del torneo funciona bien
CRUCES_RONDA_ACTUAL = [
    ("Francia", "Marruecos"),
    ("España", "Bélgica"),
    ("Noruega", "Inglaterra"),
    ("Argentina", "Suiza"),
]

# Extiende el fixture por si la ronda actual aun no aparecia
fixture_mundial = construir_fixture_mundial(fecha_inicio="2026-06-11", fecha_fin=FECHA_FIN_FIXTURE)

datos_previos_ronda = {}
for home, away in CRUCES_RONDA_ACTUAL:
    datos_previos_ronda[(home, away)] = obtener_datos_previos_partido(fixture_mundial, home, away)
    d = datos_previos_ronda[(home, away)]
    if d:
        print(f"{home} vs {away} -> sede: {d.get('stadium')} | arbitro: {d.get('referee')} | hora UTC: {d.get('startTime')}")
    else:
        print(f"{home} vs {away} -> sin datos aun (puede que el fixture no lo tenga cargado todavia)")


In [ ]:
resultados_ronda = {}

for home, away in CRUCES_RONDA_ACTUAL:
    d = datos_previos_ronda.get((home, away))
    resultados_ronda[f"{home} vs {away}"] = predecir_partido_v4(
        home, away, fixture_mundial, cache_partidos, cache_corners, promedios,
        stadium_str=d.get("stadium") if d else None,
        start_time_iso=d.get("startTime") if d else None,
        arbitro=d.get("referee") if d else None,
        perfil_arbitros=perfil_arbitros,
        es_knockout=True, alpha_incertidumbre=ALPHA_KNOCKOUT, k_fifa=K_FIFA
    )

chequeo_sensatez(resultados_ronda)


### Resumen comparativo de la ronda

In [ ]:
resumen_ronda = []
for partido, r in resultados_ronda.items():
    if r is None:
        continue
    home, away = partido.split(" vs ")
    resumen_ronda.append({
        "Partido": partido,
        "Gana local": f"{r['p_local']:.1%}",
        "Empate": f"{r['p_empate']:.1%}",
        "Gana visita": f"{r['p_visit']:.1%}",
        "Over 2.5": f"{r['over25']:.1%}",
        "BTTS": f"{r['btts']:.1%}",
    })

df_resumen_ronda = pd.DataFrame(resumen_ronda)
display(df_resumen_ronda)


### Nota honesta, bro
Entre mas avanza el torneo, menos partidos quedan para calibrar y mas parejos son los equipos — es normal ver probabilidades mas repartidas (menos favoritos con 70-80%). No es un defecto del modelo, es la realidad de que en semis/final ya no quedan equipos claramente mas debiles.

In [ ]:
import os
import json as json_lib
import psycopg2
from dotenv import load_dotenv

load_dotenv("/mnt/c/Users/Ultra 7/Desktop/PROYECTOS BI/proyecto_mundial/.env")

def a_nativo(valor):
    if valor is None:
        return None
    if hasattr(valor, "item"):
        return valor.item()
    return valor

def guardar_predicciones_db(resultados_dict, round_actual):
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )
    cur = conn.cursor()

    for partido, r in resultados_dict.items():
        if r is None:
            continue
        home, away = partido.split(" vs ")

        detalle = {
            "sede": r.get("sede"), "altitud_m": a_nativo(r.get("altitud_m")),
            "temp_prom_c": a_nativo(r.get("temp_prom_c")), "hora_local": r.get("hora_local"),
            "arbitro": r.get("arbitro"),
            "lambda_remates": a_nativo(r.get("lambda_remates")),
            "lambda_remates_arco": a_nativo(r.get("lambda_remates_arco")),
            "lambda_corners": a_nativo(r.get("lambda_corners")),
            "marcadores_top": r.get("marcadores_top", []),
            "grilla_completa": r.get("grilla_completa", []),
        }

        cur.execute("""
            INSERT INTO predicciones
                (home_team, away_team, round, prob_local, prob_empate, prob_visitante,
                 over25, btts, lambda_tarjetas, lambda_faltas, detalle, fecha_prediccion)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, NOW())
            ON CONFLICT (home_team, away_team, round) DO UPDATE SET
                prob_local = EXCLUDED.prob_local,
                prob_empate = EXCLUDED.prob_empate,
                prob_visitante = EXCLUDED.prob_visitante,
                over25 = EXCLUDED.over25,
                btts = EXCLUDED.btts,
                lambda_tarjetas = EXCLUDED.lambda_tarjetas,
                lambda_faltas = EXCLUDED.lambda_faltas,
                detalle = EXCLUDED.detalle,
                fecha_prediccion = NOW()
        """, (
            home, away, round_actual,
            a_nativo(r.get("p_local")), a_nativo(r.get("p_empate")), a_nativo(r.get("p_visit")),
            a_nativo(r.get("over25")), a_nativo(r.get("btts")),
            a_nativo(r.get("lambda_tarjetas")), a_nativo(r.get("lambda_faltas")),
            json_lib.dumps(detalle),
        ))

    conn.commit()
    cur.close()
    conn.close()
    print(f"Guardadas {len(resultados_dict)} predicciones en la base de datos.")

guardar_predicciones_db(resultados_ronda, round_actual=ROUND_ACTUAL)
